In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [0]:
# load cleaned data
df = pd.read_csv("../data/processed/cleaned_basketball_data.csv")
df.head()

In [0]:
# basic stats
print(df.describe())
print("\nData types:")
print(df.dtypes)

In [0]:
# distribution of win percentage
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.hist(df['win_percentage'], bins=30, color='skyblue', edgecolor='black')
plt.xlabel('Win %')
plt.ylabel('Count')
plt.title('Win Percentage Distribution')

plt.subplot(1, 2, 2)
plt.hist(df['net_rating'], bins=30, color='coral', edgecolor='black')
plt.xlabel('Net Rating')
plt.ylabel('Count')
plt.title('Net Rating Distribution')

plt.tight_layout()
plt.show()

In [0]:
# skewness analysis
from scipy.stats import skew

# calculate skewness for all numeric columns
skewness = df.select_dtypes(include=[np.number]).skew().sort_values(ascending=False)

print("\nInterpretation Guide:")
print("  |skew| < 0.5  : Fairly symmetric")
print("  0.5-1.0       : Moderately skewed")
print("  |skew| > 1.0  : Highly skewed\n")

# sort by absolute skewness
skewness_abs = skewness.abs().sort_values(ascending=False)
for col in skewness_abs.index:
    skew_val = skewness[col]
    abs_skew = abs(skew_val)
    
    if abs_skew < 0.5:
        level = "Fairly symmetric"
    elif abs_skew < 1.0:
        level = "Moderately skewed"
    else:
        level = "HIGHLY SKEWED"
    
    print(f"{col:30s}: {skew_val:7.3f}  [{level}]")

# identify highly skewed features
highly_skewed = skewness_abs[skewness_abs > 1.0]
print(f"\n{len(highly_skewed)} features are highly skewed (|skew| > 1.0)")
if len(highly_skewed) > 0:
    print("\nHighly skewed features:")
    for col in highly_skewed.index:
        print(f"  - {col}: {skewness[col]:.3f}")

In [0]:
# correlation with win percentage
corr_with_win = df.corr(numeric_only=True)['win_percentage'].sort_values(ascending=False)
print("Top correlations with win percentage:")
print(corr_with_win.head(10))

# plot top correlations
top_features = corr_with_win.head(8).index[1:]  # exclude win_percentage itself
plt.figure(figsize=(10, 6))
plt.barh(top_features, corr_with_win[top_features])
plt.xlabel('Correlation')
plt.title('Features Most Correlated with Winning')
plt.tight_layout()
plt.show()

In [0]:
# scatter plots - win% vs key metrics
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].scatter(df['net_rating'], df['win_percentage'], alpha=0.6)
axes[0, 0].set_xlabel('Net Rating')
axes[0, 0].set_ylabel('Win %')
axes[0, 0].set_title('Net Rating vs Win %')

axes[0, 1].scatter(df['offensive_rating'], df['win_percentage'], alpha=0.6, color='green')
axes[0, 1].set_xlabel('Offensive Rating')
axes[0, 1].set_ylabel('Win %')
axes[0, 1].set_title('Offensive Rating vs Win %')

axes[1, 0].scatter(df['defensive_rating'], df['win_percentage'], alpha=0.6, color='red')
axes[1, 0].set_xlabel('Defensive Rating')
axes[1, 0].set_ylabel('Win %')
axes[1, 0].set_title('Defensive Rating vs Win %')

axes[1, 1].scatter(df['assist_turnover_ratio'], df['win_percentage'], alpha=0.6, color='orange')
axes[1, 1].set_xlabel('Assist/Turnover Ratio')
axes[1, 1].set_ylabel('Win %')
axes[1, 1].set_title('Ball Control vs Win %')

plt.tight_layout()
plt.show()

In [0]:
# offensive vs defensive rating
plt.figure(figsize=(10, 6))
plt.scatter(df['offensive_rating'], df['defensive_rating'], 
            c=df['win_percentage'], cmap='RdYlGn', alpha=0.7, s=80)
plt.colorbar(label='Win %')
plt.xlabel('Offensive Rating')
plt.ylabel('Defensive Rating')
plt.title('Offense vs Defense (colored by win %)')
plt.axhline(df['defensive_rating'].median(), color='gray', linestyle='--', alpha=0.5)
plt.axvline(df['offensive_rating'].median(), color='gray', linestyle='--', alpha=0.5)
plt.show()

In [0]:
# compare top and bottom teams
top_teams = df.nsmallest(10, 'rank')
bottom_teams = df.nlargest(10, 'rank')

metrics = ['points', 'opponent_points', 'assists', 'turnovers', 'total_rebounds']
avg_top = top_teams[metrics].mean()
avg_bottom = bottom_teams[metrics].mean()

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - width/2, avg_top, width, label='Top 10 Teams', color='green', alpha=0.7)
ax.bar(x + width/2, avg_bottom, width, label='Bottom 10 Teams', color='red', alpha=0.7)

ax.set_xlabel('Metrics')
ax.set_ylabel('Average')
ax.set_title('Top 10 vs Bottom 10 Teams Comparison')
ax.set_xticks(x)
ax.set_xticklabels(metrics, rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

In [0]:
# home vs away performance
plt.figure(figsize=(10, 6))

plt.subplot(1, 2, 1)
plt.scatter(df['home_win_pct'], df['away_win_pct'], alpha=0.6)
plt.plot([0, 1], [0, 1], 'r--', alpha=0.5)  # diagonal line
plt.xlabel('Home Win %')
plt.ylabel('Away Win %')
plt.title('Home vs Away Performance')

plt.subplot(1, 2, 2)
plt.hist(df['home_advantage'], bins=25, color='purple', edgecolor='black', alpha=0.7)
plt.xlabel('Home Advantage (home - away win %)')
plt.ylabel('Count')
plt.title('Home Court Advantage Distribution')
plt.axvline(0, color='red', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [0]:
# correlation heatmap
plt.figure(figsize=(12, 10))
key_cols = ['win_percentage', 'net_rating', 'offensive_rating', 'defensive_rating',
            'points', 'assists', 'turnovers', 'total_rebounds', 'steals', 'blocks',
            'assist_turnover_ratio', 'scoring_efficiency']

corr = df[key_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, linewidths=0.5)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()